# Lab 05 — Attention Variants and Mixture-of-Experts
**Goal:** Compare multi-head attention with grouped-query style projections and simulate expert routing/load balance.

Free-first: run on CPU or free Colab where possible.

Method: **predict → run → inspect → break → decide**.


In [ ]:
import torch
B,T,H,D=2,16,8,64
q=torch.randn(B,H,T,D)
k_full=torch.randn(B,H,T,D)
k_group=torch.randn(B,2,T,D)  # 2 KV groups shared by query heads
print("Q:",q.shape,"full K:",k_full.shape,"grouped K:",k_group.shape)


## Predict before running
Predict whether sharing K/V across groups reduces KV-cache memory. For MoE, what happens if one expert receives most tokens?


In [ ]:
# Estimate KV-cache elements for MHA vs a 2-group design.
mha_cache=2*B*T*H*D
gqa_cache=2*B*T*2*D
print("MHA KV elements:",mha_cache)
print("GQA-like KV elements:",gqa_cache)
print("ratio:",mha_cache/gqa_cache)


## Explain the result
Grouped-query attention reduces the number of distinct K/V streams that must be cached, while preserving multiple query heads. MoE can increase parameter capacity without activating every expert for every token, but routing imbalance wastes capacity and can create communication hotspots.


In [ ]:
import torch.nn.functional as F
router=torch.randn(1000,8)
p=F.softmax(router,dim=-1)
top=p.argmax(-1)
counts=torch.bincount(top,minlength=8).float()
print("expert fractions:",(counts/counts.sum()).tolist())
print("max/min load ratio:",float(counts.max()/counts.min().clamp_min(1)))


## Critical-thinking answers
The correct architecture choice depends on the serving and training target. A method that saves memory can still hurt quality or create load-balancing complexity. Measure end-to-end behavior, not one local tensor.

### Sources
https://cs336.stanford.edu/
